In [81]:
import optuna

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [31]:
df = pd.read_csv(r"00 pima india diabetes dataset.csv")

# The problem with this dataset is, here null values are represented with zero
# So first we will convert these 0 with np.nan and then fill those values

In [32]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [36]:
columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

df[columns] = df[columns].replace(0, np.nan)

In [38]:
df.isnull().sum()

Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

In [41]:
# Filling with mean()

df.fillna(df.mean(), inplace = True)

df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [43]:
# train_test_split

x = df.drop('Outcome', axis = 'columns')
y = df.drop(x, axis = 'columns')

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 1)

In [45]:
# Scaling

ss = StandardScaler()

x_train_trans = ss.fit_transform(x_train)
x_test_trans = ss.transform(x_test)

print(x_train_trans.shape)
print(x_test_trans.shape)

(614, 8)
(154, 8)


# Starting optuna:

It's a very easy 3 steps or can say 3 lines of code. The working is same as kerastuner in CNN.

In [79]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [80]:
def objective(trial):

    # Suggest values for hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 250)    # Select from 50 to 250
    max_depth = trial.suggest_int('max_depth', 3, 20)    # select from 3 to 20
    criterion = trial.suggest_categorical('criterion', ['entropy', 'gini', 'log_loss'])

    # Create RandomForestClassifier with suggested hyp

    rf = RandomForestClassifier(
        n_estimators = n_estimators,
        max_depth = max_depth,
        criterion = criterion
    )

    # Performing 5-fold cross_validation

    acc = cross_val_score(
        estimator = rf,
        X = x_train_trans,
        y = y_train,
        scoring = 'accuracy',
        cv = 5,
        n_jobs = -1
    ).mean()

    return acc

In [85]:
study = optuna.create_study(direction = 'maximize', sampler = optuna.samplers.TPESampler(seed = 1))
study.optimize(func = objective, n_trials = 50, n_jobs = -1)

# We are saying to optimizer() that this (objective()) is the process of finding accuracy, do it for 50 trials 

# direction = 'maximize' bcz we are using accuracy
# sampler = optuna.samplers.TPESampler() by-default

[I 2025-05-03 22:48:30,467] A new study created in memory with name: no-name-6e2b596a-5fe1-431f-b72b-a9b2a28a77c7
[I 2025-05-03 22:48:30,733] Trial 1 finished with value: 0.7622417699586832 and parameters: {'n_estimators': 86, 'max_depth': 3, 'criterion': 'log_loss'}. Best is trial 1 with value: 0.7622417699586832.
[I 2025-05-03 22:48:30,877] Trial 4 finished with value: 0.7654404904704785 and parameters: {'n_estimators': 58, 'max_depth': 14, 'criterion': 'entropy'}. Best is trial 4 with value: 0.7654404904704785.
[I 2025-05-03 22:48:30,960] Trial 2 finished with value: 0.7638544582167132 and parameters: {'n_estimators': 76, 'max_depth': 11, 'criterion': 'entropy'}. Best is trial 4 with value: 0.7654404904704785.
[I 2025-05-03 22:48:31,159] Trial 5 finished with value: 0.7638278022124483 and parameters: {'n_estimators': 104, 'max_depth': 6, 'criterion': 'log_loss'}. Best is trial 4 with value: 0.7654404904704785.
[I 2025-05-03 22:48:31,575] Trial 3 finished with value: 0.76385445821671

In [87]:
print(f'Best accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_params}')

Best accuracy: 0.773610555777689
Best hyperparameters: {'n_estimators': 126, 'max_depth': 4, 'criterion': 'log_loss'}


# Other samplers in optuna:

1) optuna.samplers.TPESampler()
2) `For RandomSearchCV`: optuna.samplers.RandomSampler()

3) `GridSearch`

search_space = {
'n_estimator': [7, 9, 11, 15],
'max_depth': [3,4,5,6,7]
}

--> optuna.samplers.GridSampler(search_space)